In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split , GridSearchCV , KFold , cross_val_score
from sklearn.linear_model import LinearRegression 
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score , mean_squared_error , mean_absolute_error
from sklearn.compose import ColumnTransformer , TransformedTargetRegressor
from sklearn.pipeline import Pipeline 
from sklearn.preprocessing import StandardScaler , OneHotEncoder , OrdinalEncoder , PowerTransformer
from sklearn.base import BaseEstimator , TransformerMixin
import optuna
from lightgbm import LGBMRegressor
import mlflow
import dagshub
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_parallel_coordinate , plot_slice

C:\Users\LeoML\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dagshub.init(repo_owner="mridul0010" , repo_name="Delivery-Time-Analysis-Prediction" , mlflow=True)

Accessing as mridul0010

Initialized MLflow to track repo "mridul0010/Delivery-Time-Analysis-Prediction"

Repository mridul0010/Delivery-Time-Analysis-Prediction initialized!

In [3]:
mlflow.set_tracking_uri("https://dagshub.com/mridul0010/Delivery-Time-Analysis-Prediction.mlflow")

In [4]:
mlflow.set_experiment("Model Selection")

2026/07/01 22:40:18 INFO mlflow.tracking.fluent: Experiment with name 'Model Selection' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/1f45928a57534e5e9119d535a3d497a7', creation_time=1782925820593, effective_trace_archival_retention=None, experiment_id='5', last_update_time=1782925820593, lifecycle_stage='active', name='Model Selection', tags={}, trace_location=None, workspace='default'>

In [ ]:
df = pd.read_csv('../data/Cleaned Delivery Dataset.csv')

In [6]:
pd.set_option('display.max_columns' , None)

In [7]:
df.drop(columns=['Delivery_Agent'] , inplace=True)

In [8]:
df['Order_Datetime'] = pd.to_datetime(df['Order_Datetime'])
df['Pickup_Datetime'] = pd.to_datetime(df['Pickup_Datetime'])

In [9]:
class FeatureEngineering(BaseEstimator, TransformerMixin):
    
    def __init__(self):
        self.rating_bins = None

    def fit(self, X, y=None):
        X = X.copy()
        
        # Store bins for consistent transformation
        self.rating_bins = pd.qcut(
            X['Delivery_person_Ratings'],
            q=3,
            retbins=True,
            duplicates='drop'
        )[1]
        
        return self

    def transform(self, X):
        X = X.copy()

        # Datetime conversion
        X['Order_Datetime'] = pd.to_datetime(X['Order_Datetime'], errors='coerce')
        X['Pickup_Datetime'] = pd.to_datetime(X['Pickup_Datetime'], errors='coerce')

        # Rating group
        X["delivery_rating_group"] = pd.cut(
            X['Delivery_person_Ratings'],
            bins=self.rating_bins,
            labels=['Low', 'Medium', 'High'],
            include_lowest=True
        )

        # Age group
        X["age_group"] = pd.cut(
            X['Delivery_person_Age'],
            bins=[14, 25, 35, 60],
            labels=['Young', 'Adult', 'Senior']
        )

        # Distance group
        X["distance_group"] = pd.cut(
            X['distance_km'],
            bins=[0, 5, 10, 25],
            labels=['Short Distance', 'Medium Distance', 'Long Distance']
        )

        # Time features
        X['Prep_Time(min)'] = (
            X['Pickup_Datetime'] - X['Order_Datetime']
        ).dt.total_seconds() / 60

        X['Order_hour'] = X['Order_Datetime'].dt.hour
        X['Order_day'] = X['Order_Datetime'].dt.day_name()

        X['isWeekend'] = X['Order_day'].isin(["Saturday", "Sunday"]).astype(int)

        X['Time_Of_Day'] = pd.cut(
            X['Order_hour'],
            bins=[0, 6, 12, 18, 24],
            labels=["Night", "Morning", "Afternoon", "Evening"],
            include_lowest=True
        )

        # Drop raw datetime
        X = X.drop(columns=['Order_Datetime', 'Pickup_Datetime'], errors='ignore')

        return X

    def get_feature_names_out(self , input_features = None):
        return input_features

In [10]:
X = df.drop(columns=['Time_taken (min)'])
y = df['Time_taken (min)']

In [11]:
X_train , X_test , y_train , y_test = train_test_split(X , y , test_size=0.2 , random_state=42)

In [12]:
X

,City,Zone,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,distance_km,Order_Datetime,Pickup_Datetime,Weather_conditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City_Type
0,Dehradun,Zone17,36,4.2,30.327968,78.046106,30.397968,78.116106,10.28,2022-12-02 21:55:00,2022-12-02 22:10:00,Fog,Jam,Good,Snack,motorcycle,3,No,Metropolitian
1,Kochi,Zone16,21,4.7,10.003064,76.307589,10.043064,76.347589,6.24,2022-02-13 14:55:00,2022-02-13 15:05:00,Stormy,High,Average,Meal,motorcycle,1,No,Metropolitian
2,Pune,Zone13,23,4.7,18.562450,73.916619,18.652450,74.006619,13.79,2022-04-03 17:30:00,2022-04-03 17:40:00,Sandstorms,Medium,Average,Drinks,scooter,1,No,Metropolitian
3,Ludhiana,Zone15,34,4.3,30.899584,75.809346,30.919584,75.829346,2.93,2022-02-13 09:20:00,2022-02-13 09:30:00,Sandstorms,Low,poor,Buffet,motorcycle,0,No,Metropolitian
4,Kanpur,Zone14,24,4.7,26.463504,80.372929,26.593504,80.502929,19.40,2022-02-14 19:50:00,2022-02-14 20:05:00,Fog,Jam,Average,Snack,scooter,1,No,Metropolitian
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39637,Bangalore,Zone16,28,4.9,13.029198,77.570997,13.059198,77.600997,4.66,2022-03-30 21:55:00,2022-03-30 22:05:00,Sandstorms,Jam,Average,Meal,scooter,1,No,Metropolitian
39638,Ranchi,Zone16,35,4.2,23.371292,85.327872,23.481292,85.437872,16.60,2022-08-03 21:45:00,2022-08-03 21:55:00,Windy,Jam,Good,Drinks,motorcycle,1,No,Metropolitian
39639,Chennai,Zone08,30,4.9,13.022394,80.242439,13.052394,80.272439,4.66,2022-11-03 23:50:00,2022-11-04 00:05:00,Cloudy,Low,Average,Drinks,scooter,0,No,Metropolitian
39640,Coimbatore,Zone11,20,4.7,11.001753,76.986241,11.041753,77.026241,6.23,2022-07-03 13:35:00,2022-07-03 13:40:00,Cloudy,High,poor,Snack,motorcycle,1,No,Metropolitian


In [13]:
numerical = ['Delivery_person_Age', 'Delivery_person_Ratings', 'Restaurant_latitude',
       'Restaurant_longitude', 'Delivery_location_latitude',
       'Delivery_location_longitude', 
       'multiple_deliveries', 'distance_km', 'Prep_Time(min)', 'Order_hour',
       'isWeekend']

ohe_col = ['City', 'Zone', 'Weather_conditions',
       'Type_of_order', 'Type_of_vehicle',
       'City_Type','Order_day', 'Time_Of_Day']

ordinal_col = ['Road_traffic_density' , 'Vehicle_condition' , 'Festival',
       'delivery_rating_group' , 'age_group' , 'distance_group']

In [ ]:
Road_traffic_density = ['Low', 'Medium', 'High', 'Jam']
Vehicle_condition = ['poor', 'Average', 'Good' , 'Excellent']
Festival = ['No', 'Yes']
delivery_rating_group = ['Low', 'Medium', 'High']
age_group = ['Young', 'Adult', 'Senior']         
distance_group = ['Short Distance', 'Medium Distance', 'Long Distance']

In [ ]:
transformer = ColumnTransformer(
    transformers=[
        ("ohe" , OneHotEncoder(sparse_output=False , handle_unknown='ignore') , ohe_col),
        ('oe' , OrdinalEncoder(categories=[
            Road_traffic_density , Vehicle_condition,
            Festival , delivery_rating_group,
            age_group , distance_group
        ]) , ordinal_col),
        ('Scaling' , StandardScaler() , numerical)
    ],remainder='passthrough'
)

In [15]:
pipeline = Pipeline(
    [
        ("FeatureEngineering" , FeatureEngineering()),
        ("ColumnTransformer" , transformer)
    ]
) 

In [16]:
X_train_trans = pipeline.fit_transform(X_train)
X_test_trans = pipeline.transform(X_test)

In [17]:
pt = PowerTransformer()

In [18]:
def objective(trial):
    with mlflow.start_run(nested=True):
        model_name = trial.suggest_categorical("model",["SVM","RF","KNN","GB","XGB","LGBM"])

        if model_name == "SVM":
            kernel_svm = trial.suggest_categorical("kernel_svm",["linear","poly","rbf"])
            if kernel_svm == "linear":
                c_linear = trial.suggest_float("c_linear",0,10)
                model = SVR(C=c_linear,kernel="linear")

            elif kernel_svm == "poly":
                c_poly = trial.suggest_float("c_poly",0,10)
                degree_poly = trial.suggest_int("degree_poly",1,5)
                model = SVR(C=c_poly,degree=degree_poly,
                            kernel="poly")

            else:
                c_rbf = trial.suggest_float("c_rbf",0,100)
                gamma_rbf = trial.suggest_float("gamma_rbf",0,10)
                model = SVR(C=c_rbf,gamma=gamma_rbf,
                            kernel="rbf")

        elif model_name == "RF":
            n_estimators_rf = trial.suggest_int("n_estimators_rf",10,200)
            max_depth_rf = trial.suggest_int("max_depth_rf",2,20)
            rf = RandomForestRegressor(n_estimators=n_estimators_rf,
                                        max_depth=max_depth_rf,
                                        random_state=42,
                                        n_jobs=-1)
            model = TransformedTargetRegressor(regressor=rf , transformer=pt)

        elif model_name == "GB":
            n_estimators_gb = trial.suggest_int("n_estimators_gb",10,200)
            learning_rate_gb = trial.suggest_float("learning_rate_gb",0,1)
            max_depth_gb = trial.suggest_int("max_depth_gb",2,20)
            gb = GradientBoostingRegressor(n_estimators=n_estimators_gb,
                                                learning_rate=learning_rate_gb,
                                                max_depth=max_depth_gb,
                                                random_state=42)
            model = TransformedTargetRegressor(regressor=gb , transformer=pt)

        elif model_name == "KNN":
            n_neighbors_knn = trial.suggest_int("n_neighbors_knn",1,25)
            weights_knn = trial.suggest_categorical("weights_knn",["uniform","distance"])
            knn = KNeighborsRegressor(n_neighbors=n_neighbors_knn,
                                        weights=weights_knn,n_jobs=-1)
            model = TransformedTargetRegressor(regressor=knn , transformer=pt)

        elif model_name == "XGB":
            n_estimators_xgb = trial.suggest_int("n_estimators_xgb",10,200)
            learning_rate_xgb = trial.suggest_float("learning_rate_xgb",0.1,0.5)
            max_depth_xgb = trial.suggest_int("max_depth_xgb",2,20)
            xgb = XGBRegressor(n_estimators=n_estimators_xgb,
                                    learning_rate=learning_rate_xgb,
                                    max_depth=max_depth_xgb,
                                    random_state=42,
                                    n_jobs=-1)
            model = TransformedTargetRegressor(regressor=xgb , transformer=pt)

        elif model_name == "LGBM":
            n_estimators_lgbm = trial.suggest_int("n_estimators_lgbm",10,200)
            learning_rate_lgbm = trial.suggest_float("learning_rate_lgbm",0.1,0.5)
            max_depth_lgbm = trial.suggest_int("max_depth_lgbm",2,20)
            lgbm = LGBMRegressor(n_estimators=n_estimators_lgbm,
                                    learning_rate=learning_rate_lgbm,
                                    max_depth=max_depth_lgbm,
                                    random_state=42)
            model = TransformedTargetRegressor(regressor=lgbm , transformer=pt)


        model.fit(X_train_trans , y_train)


        # log params
        mlflow.log_params(trial.params)
        mlflow.log_param("model_type" , model_name)

        

        # get the predictions
        y_pred_test = model.predict(X_test_trans)

        # calculate the error
        error = mean_absolute_error(y_test,y_pred_test)

        # log model_name
        mlflow.log_param("model",model_name)

        # log error
        mlflow.log_metric("MAE",error)

        return error

In [ ]:
study = optuna.create_study(direction='minimize')

with mlflow.start_run(run_name="best_model"):
    # optimize the objective function
    study.optimize(objective , n_trials=50 , n_jobs=-1 , show_progress_bar=True)

    # log the best params
    mlflow.log_params(study.best_params)

    # log best score
    mlflow.log_metric("best_score" , study.best_value)

    # training the lgbm on best param
    best_xgb = XGBRegressor(**study.best_params)

    best_model = TransformedTargetRegressor(regressor=best_xgb , transformer=pt)

    best_model.fit(X_train_trans , y_train)

    y_pred_train = best_model.predict(X_train_trans)
    y_pred_test = best_model.predict(X_test_trans)

    scores = cross_val_score(
        best_model,
        X_train_trans,
        y_train,
        scoring="neg_mean_absolute_error",
        cv=5,n_jobs=-1
    )

    # logging metrics
    mlflow.log_metric("Training_error" ,mean_absolute_error(y_train ,y_pred_train))
    mlflow.log_metric("Test_error" ,mean_absolute_error(y_test ,y_pred_test))
    mlflow.log_metric("Training_r2" ,r2_score(y_train ,y_pred_train))
    mlflow.log_metric("Test_r2" ,r2_score(y_test ,y_pred_test))
    mlflow.log_metric("cross_val" , -scores.mean())
    
    # log the best model 
    mlflow.sklearn.log_model(best_xgb , artifact_path="Model_Selection")

[I 2026-07-01 22:40:20,728] A new study created in memory with name: no-name-85f47f91-c230-47fb-ae17-dd304338339a
  0%|          | 0/50 [00:00<?, ?it/s]

🏃 View run honorable-hare-937 at: https://dagshub.com/mridul0010/Delivery-Time-Analysis-Prediction.mlflow/#/experiments/5/runs/ee065c9adc194220a1c291b756fdb1bd
🧪 View experiment at: https://dagshub.com/mridul0010/Delivery-Time-Analysis-Prediction.mlflow/#/experiments/5
🏃 View run gifted-hawk-806 at: https://dagshub.com/mridul0010/Delivery-Time-Analysis-Prediction.mlflow/#/experiments/5/runs/b6641c78d75c4053a8cd841461af33c7
🧪 View experiment at: https://dagshub.com/mridul0010/Delivery-Time-Analysis-Prediction.mlflow/#/experiments/5
🏃 View run likeable-bird-78 at: https://dagshub.com/mridul0010/Delivery-Time-Analysis-Prediction.mlflow/#/experiments/5/runs/45399be25b6344b9a2d828f70f381e14
🧪 View experiment at: https://dagshub.com/mridul0010/Delivery-Time-Analysis-Prediction.mlflow/#/experiments/5


Best trial: 15. Best value: 3.0547:   2%|▏         | 1/50 [00:48<39:28, 48.34s/it]

[I 2026-07-01 22:41:09,889] Trial 15 finished with value: 3.0546997090644923 and parameters: {'model': 'RF', 'n_estimators_rf': 68, 'max_depth_rf': 13}. Best is trial 15 with value: 3.0546997090644923.


Best trial: 15. Best value: 3.0547:   4%|▍         | 2/50 [00:52<17:49, 22.28s/it]

...

[I 2026-07-01 23:05:06,593] Trial 16 finished with value: 4.062758319094305 and parameters: {'model': 'SVM', 'kernel_svm': 'poly', 'c_poly': 5.5217352840803455, 'degree_poly': 5}. Best is trial 8 with value: 3.028179407119751.
🏃 View run angry-mink-827 at: https://dagshub.com/mridul0010/Delivery-Time-Analysis-Prediction.mlflow/#/experiments/5/runs/64c5cd47a11f4d129cf7974c6123dad8
🧪 View experiment at: https://dagshub.com/mridul0010/Delivery-Time-Analysis-Prediction.mlflow/#/experiments/5


Best trial: 8. Best value: 3.02818: 100%|██████████| 50/50 [27:45<00:00, 33.31s/it]


[I 2026-07-01 23:08:06,816] Trial 41 finished with value: 4.096670094198045 and parameters: {'model': 'SVM', 'kernel_svm': 'poly', 'c_poly': 7.981483589730658, 'degree_poly': 5}. Best is trial 8 with value: 3.028179407119751.


C:\Users\LeoML\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\xgboost\training.py:199: UserWarning: [23:08:08] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "learning_rate_xgb", "max_depth_xgb", "model", "n_estimators_xgb" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
[W 2026-07-01 23:08:13,919] Your study has only completed trials with missing parameters.
2026/07/01 23:08:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/01 23:08:21 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run best_model at: https://dagshub.com/mridul0010/Delivery-Time-Analysis-Prediction.mlflow/#/experiments/5/runs/b5e247c29c4b4d20814384191a1efcac
🧪 View experiment at: https://dagshub.com/mridul0010/Delivery-Time-Analysis-Prediction.mlflow/#/experiments/5


In [20]:
study.best_value

3.028179407119751

In [22]:
study.best_params

{'model': 'XGB',
 'n_estimators_xgb': 19,
 'learning_rate_xgb': 0.24015669612328497,
 'max_depth_xgb': 10}

In [25]:
study.trials_dataframe().head()

,number,value,datetime_start,datetime_complete,duration,params_c_linear,params_c_poly,params_c_rbf,params_degree_poly,params_gamma_rbf,params_kernel_svm,params_learning_rate_gb,params_learning_rate_lgbm,params_learning_rate_xgb,params_max_depth_gb,params_max_depth_lgbm,params_max_depth_rf,params_max_depth_xgb,params_model,params_n_estimators_gb,params_n_estimators_lgbm,params_n_estimators_rf,params_n_estimators_xgb,params_n_neighbors_knn,params_weights_knn,state
0,0,3.264776,2026-07-01 22:40:21.558746,2026-07-01 22:42:03.484280,0 days 00:01:41.925534,NaN,NaN,NaN,NaN,NaN,NaN,0.634651,NaN,NaN,6.0,NaN,NaN,NaN,GB,17.0,NaN,NaN,NaN,NaN,NaN,COMPLETE
1,1,4.076419,2026-07-01 22:40:21.559535,2026-07-01 22:45:19.742331,0 days 00:04:58.182796,NaN,NaN,NaN,NaN,NaN,NaN,0.908509,NaN,NaN,12.0,NaN,NaN,NaN,GB,180.0,NaN,NaN,NaN,NaN,NaN,COMPLETE
2,2,4.464248,2026-07-01 22:40:21.562678,2026-07-01 22:41:53.756217,0 days 00:01:32.193539,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,KNN,NaN,NaN,NaN,NaN,17.0,distance,COMPLETE
3,3,5.807351,2026-07-01 22:40:21.564946,2026-07-01 22:41:25.801324,0 days 00:01:04.236378,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN,RF,NaN,NaN,138.0,NaN,NaN,NaN,COMPLETE
4,4,4.562442,2026-07-01 22:40:21.569808,2026-07-01 22:41:39.471416,0 days 00:01:17.901608,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,KNN,NaN,NaN,NaN,NaN,6.0,distance,COMPLETE


In [27]:
study.trials_dataframe()['params_model'].value_counts()

params_model
SVM     17
XGB     13
RF       6
GB       5
KNN      5
LGBM     4
Name: count, dtype: int64

In [29]:
best_xgb = XGBRegressor(**study.best_params)

model = TransformedTargetRegressor(regressor=best_xgb , transformer=pt)

model.fit(X_train_trans , y_train)

C:\Users\LeoML\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\xgboost\training.py:199: UserWarning:

[23:26:34] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "learning_rate_xgb", "max_depth_xgb", "model", "n_estimators_xgb" } are not used.




,"regressor regressor: object, default=NoneRegressor object such as derived from:class:`~sklearn.base.RegressorMixin`. This regressor willautomatically be cloned each time prior to fitting. If `regressor isNone`, :class:`~sklearn.linear_model.LinearRegression` is created and used.","XGBRegressor(...egy=None, ...)"
,"transformer transformer: object, default=NoneEstimator object such as derived from:class:`~sklearn.base.TransformerMixin`. Cannot be set at the same timeas `func` and `inverse_func`. If `transformer is None` as well as`func` and `inverse_func`, the transformer will be an identitytransformer. Note that the transformer will be cloned during fitting.Also, the transformer is restricting `y` to be a numpy array.",PowerTransformer()
,"func func: function, default=NoneFunction to apply to `y` before passing to :meth:`fit`. Cannot be setat the same time as `transformer`. If `func is None`, the function used will bethe identity function. If `func` is set, `inverse_func` also needs to beprovided. The function needs to return a 2-dimensional array.",None
,"inverse_func inverse_func: function, default=NoneFunction to apply to the prediction of the regressor. Cannot be set atthe same time as `transformer`. The inverse function is used to returnpredictions to the same space of the original training labels. If`inverse_func` is set, `func` also needs to be provided. The inversefunction needs to return a 2-dimensional array.",None
,"check_inverse check_inverse: bool, default=TrueWhether to check that `transform` followed by `inverse_transform`or `func` followed by `inverse_func` leads to the original targets.",True
,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None


In [30]:
y_pred = model.predict(X_test_trans)

r2_score(y_test , y_pred)

0.8177204132080078

In [31]:
study.trials_dataframe().head()

,number,value,datetime_start,datetime_complete,duration,params_c_linear,params_c_poly,params_c_rbf,params_degree_poly,params_gamma_rbf,params_kernel_svm,params_learning_rate_gb,params_learning_rate_lgbm,params_learning_rate_xgb,params_max_depth_gb,params_max_depth_lgbm,params_max_depth_rf,params_max_depth_xgb,params_model,params_n_estimators_gb,params_n_estimators_lgbm,params_n_estimators_rf,params_n_estimators_xgb,params_n_neighbors_knn,params_weights_knn,state
0,0,3.264776,2026-07-01 22:40:21.558746,2026-07-01 22:42:03.484280,0 days 00:01:41.925534,NaN,NaN,NaN,NaN,NaN,NaN,0.634651,NaN,NaN,6.0,NaN,NaN,NaN,GB,17.0,NaN,NaN,NaN,NaN,NaN,COMPLETE
1,1,4.076419,2026-07-01 22:40:21.559535,2026-07-01 22:45:19.742331,0 days 00:04:58.182796,NaN,NaN,NaN,NaN,NaN,NaN,0.908509,NaN,NaN,12.0,NaN,NaN,NaN,GB,180.0,NaN,NaN,NaN,NaN,NaN,COMPLETE
2,2,4.464248,2026-07-01 22:40:21.562678,2026-07-01 22:41:53.756217,0 days 00:01:32.193539,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,KNN,NaN,NaN,NaN,NaN,17.0,distance,COMPLETE
3,3,5.807351,2026-07-01 22:40:21.564946,2026-07-01 22:41:25.801324,0 days 00:01:04.236378,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN,RF,NaN,NaN,138.0,NaN,NaN,NaN,COMPLETE
4,4,4.562442,2026-07-01 22:40:21.569808,2026-07-01 22:41:39.471416,0 days 00:01:17.901608,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,KNN,NaN,NaN,NaN,NaN,6.0,distance,COMPLETE


In [32]:
plot_optimization_history(study)